# Problem Sheet 2

Loan Approval Classification


In this problem sheet, I work on a classification problem for loan approval. I import the data, do EDA, prepare the features, train several models, and compare their performance.


## 1. Import the Data

Import the loan approval data for clients (`loan_approval_dataset.csv`) as Pandas DataFrame into your notebook, name it as `loan_data`. Are there any missing data? Which features are numerical, and which features are categorical?


In [31]:
import pandas as pd
import numpy as np

loan_data = pd.read_csv("loan_approval_dataset.csv")
loan_data.columns = loan_data.columns.str.strip()
for column in loan_data.select_dtypes(include="object").columns:
    loan_data[column] = loan_data[column].str.strip()

numerical_features = loan_data.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = loan_data.select_dtypes(include=["object"]).columns.tolist()

print("Shape of the dataset:", loan_data.shape, "\n")
print("First 5 rows:")
display(loan_data.head())

missing = loan_data.isnull().sum().reset_index()
print("Missing values in each column:")
print(missing , "\n")

loan_data.info()
loan_data.nunique()

Shape of the dataset: (4269, 13) 

First 5 rows:


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


Missing values in each column:
                       index  0
0                    loan_id  0
1           no_of_dependents  0
2                  education  0
3              self_employed  0
4               income_annum  0
5                loan_amount  0
6                  loan_term  0
7                cibil_score  0
8   residential_assets_value  0
9    commercial_assets_value  0
10       luxury_assets_value  0
11          bank_asset_value  0
12               loan_status  0 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   loan_id                   4269 non-null   int64 
 1   no_of_dependents          4269 non-null   int64 
 2   education                 4269 non-null   object
 3   self_employed             4269 non-null   object
 4   income_annum              4269 non-null   int64 
 5   loan_amount               426

loan_id                     4269
no_of_dependents               6
education                      2
self_employed                  2
income_annum                  98
loan_amount                  378
loan_term                     10
cibil_score                  601
residential_assets_value     278
commercial_assets_value      188
luxury_assets_value          379
bank_asset_value             146
loan_status                    2
dtype: int64

### Answer 1: 
1. The dataset has 4269 rows and 13 columns.
2. The dataset has no mossing value.
3. `education`, `self_employed`, `loan_status` are categorical data, others are numerical data.

## 2. EDA

Perform the EDA.


In [35]:
print("Summary statistics for numerical columns:")
numerical_summary = loan_data[numerical_features].describe().round(2)
display(numerical_summary)

print("Summary statistics for categorical columns:")
categorical_summary = loan_data[categorical_features].describe()
display(categorical_summary)

loan_status_distribution = pd.DataFrame({
    "Loan Status": loan_data["loan_status"].value_counts().index,
    "Count": loan_data["loan_status"].value_counts().values,
    "Percentage": loan_data["loan_status"].value_counts(normalize=True).mul(100).round(2).values
})
print("Loan status distribution:")
display(loan_status_distribution)

important_numerical_cols = [
    "income_annum", "loan_amount", "loan_term", "cibil_score",
    "residential_assets_value", "commercial_assets_value",
    "luxury_assets_value", "bank_asset_value"
]
print("Median important numerical values by loan status:")
median_by_status = loan_data.groupby("loan_status")[important_numerical_cols].median().round(2)
display(median_by_status)

print("Correlation for selected important numerical variables:")
selected_corr_cols = ["income_annum", "loan_amount", "loan_term", "cibil_score", "bank_asset_value"]
correlation_table = loan_data[selected_corr_cols].corr().round(2)
display(correlation_table)

Summary statistics for numerical columns:


,loan_id,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.0,4269.0,4269.00,4269.00,4269.00,4269.00,4269.00,4269.00,4269.00,4269.00
mean,2135.0,2.5,5059123.92,15133450.46,10.90,599.94,7472616.54,4973155.31,15126305.93,4976692.43
std,1232.5,1.7,2806839.83,9043362.98,5.71,172.43,6503636.59,4388966.09,9103753.67,3250185.31
min,1.0,0.0,200000.00,300000.00,2.00,300.00,-100000.00,0.00,300000.00,0.00
25%,1068.0,1.0,2700000.00,7700000.00,6.00,453.00,2200000.00,1300000.00,7500000.00,2300000.00
50%,2135.0,3.0,5100000.00,14500000.00,10.00,600.00,5600000.00,3700000.00,14600000.00,4600000.00
75%,3202.0,4.0,7500000.00,21500000.00,16.00,748.00,11300000.00,7600000.00,21700000.00,7100000.00
max,4269.0,5.0,9900000.00,39500000.00,20.00,900.00,29100000.00,19400000.00,39200000.00,14700000.00


Summary statistics for categorical columns:


,education,self_employed,loan_status
count,4269,4269,4269
unique,2,2,2
top,Graduate,Yes,Approved
freq,2144,2150,2656


Loan status distribution:


,Loan Status,Count,Percentage
0,Approved,2656,62.22
1,Rejected,1613,37.78


Median important numerical values by loan status:


,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
loan_status,,,,,,,,
Approved,5000000.0,14600000.0,10.0,711.0,5400000.0,3700000.0,14400000.0,4500000.0
Rejected,5100000.0,14500000.0,12.0,429.0,5900000.0,3700000.0,14800000.0,4600000.0


Correlation for selected important numerical variables:


,income_annum,loan_amount,loan_term,cibil_score,bank_asset_value
income_annum,1.00,0.93,0.01,-0.02,0.85
loan_amount,0.93,1.00,0.01,-0.02,0.79
loan_term,0.01,0.01,1.00,0.01,0.02
cibil_score,-0.02,-0.02,0.01,1.00,-0.02
bank_asset_value,0.85,0.79,0.02,-0.02,1.00


### Answer 2:
1. I recorded some basic imnformation of numerical and categorical data.
2. I wrote the count and percentage of two loan status.
3. I used the median value to compare the approved and rejected loans. I did not use the mean because some values, such as income and asset value, can be very high or very low. These extreme values may affect the mean, but the median is more stable.
4. I calculated the correlation between `income_annum`, `loan_amount`, `loan_term`, `cibil_score`, `bank_asset_value`.

## 3. Pipeline

Implement a pipeline which drops `loan_id`, encodes categorical data, and scales the numerical attributes by standardization.


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

X = loan_data.drop(columns=["loan_id", "loan_status"])
y = loan_data["loan_status"].str.strip().map({"Rejected": 0, "Approved": 1})

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)
print("Target labels:", y.value_counts().to_dict())

Categorical columns: ['education', 'self_employed']
Numerical columns: ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']
Target labels: {1: 2656, 0: 1613}


### Answer 3:
1. I dropped two columns in features: `loan_id`(it is not used in predicting), `loan_status`(it is label)
2. I created X(feature data) and y(label)
3. I used ColumnTransformer to deal with numerical and categorical data seperately
4. For numerical data, I just standard these data(mean=0, std=1)
5. For categorical data, I use OneHotEncoder to change these data into numbers

## 4. Train-Test Split

Split the data into training data (80%) and testing data (20%).


In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).round(3))
print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).round(3))

Training data shape: (3415, 11)
Testing data shape: (854, 11)

Training target distribution:


loan_status
1    0.622
0    0.378
Name: proportion, dtype: float64


Testing target distribution:


loan_status
1    0.622
0    0.378
Name: proportion, dtype: float64

### Answer 4:
* I split the training and testing data, and keep the class distribution similar in both training and testing data by using `stratify=y`.

## 5. Classifiers

Apply the following classifiers: Linear Regression, SVM, Decision Tree, Random Forest, and KNN. For each model, provide Accuracy, Precision, Recall, F1 Score, and Classification Report.


In [42]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

class LinearRegressionClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def fit(self, X, y):
        self.model_ = LinearRegression()
        self.model_.fit(X, y)
        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        predictions = self.model_.predict(X)
        return (predictions >= self.threshold).astype(int)    # >=0.5 ----> 1; <0.5 ----> 0

models = {
    "Linear Regression": LinearRegressionClassifier(),
    "SVM": SVC(kernel="rbf", random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []
classification_reports = {}
fitted_models = {}

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    fitted_models[model_name] = clf

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })
    classification_reports[model_name] = pd.DataFrame(
        classification_report(
            y_test,
            y_pred,
            target_names=["Rejected", "Approved"],
            output_dict=True
        )
    ).round(3)

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
print("Model comparison:")
display(results_df.round(4))

for model_name, report_table in classification_reports.items():
    print()
    print("Classification Report - " + model_name)
    display(report_table)


Model comparison:


,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.9801,0.9831,0.9849,0.9840
1,Decision Tree,0.9789,0.9831,0.9831,0.9831
2,SVM,0.9461,0.9584,0.9548,0.9566
3,Linear Regression,0.9286,0.9537,0.9303,0.9418
4,KNN,0.8958,0.9108,0.9228,0.9167



Classification Report - Linear Regression


,Rejected,Approved,accuracy,macro avg,weighted avg
precision,0.890,0.954,0.929,0.922,0.930
recall,0.926,0.930,0.929,0.928,0.929
f1-score,0.907,0.942,0.929,0.925,0.929
support,323.000,531.000,0.929,854.000,854.000



Classification Report - SVM


...

,Rejected,Approved,accuracy,macro avg,weighted avg
precision,0.972,0.983,0.979,0.978,0.979
recall,0.972,0.983,0.979,0.978,0.979
f1-score,0.972,0.983,0.979,0.978,0.979
support,323.000,531.000,0.979,854.000,854.000



Classification Report - Random Forest


,Rejected,Approved,accuracy,macro avg,weighted avg
precision,0.975,0.983,0.98,0.979,0.98
recall,0.972,0.985,0.98,0.979,0.98
f1-score,0.974,0.984,0.98,0.979,0.98
support,323.000,531.000,0.98,854.000,854.00



Classification Report - KNN


,Rejected,Approved,accuracy,macro avg,weighted avg
precision,0.870,0.911,0.896,0.891,0.895
recall,0.851,0.923,0.896,0.887,0.896
f1-score,0.861,0.917,0.896,0.889,0.896
support,323.000,531.000,0.896,854.000,854.000


### Answer 3:
1. I create a class for linear regression, because originally linear regression is a regression model, now i need to put it into for loop and make those 5 model operate together. So i need to create a new class to change the syntax to put linear regression model into the for loop.
2. I write a for loop to fit these 5 model, this is more convenient than write 5 model seperately.
3. I write report for each model so that i can compare the behavier of each model easily.

## 6. Hard Voting Classifier

Apply the ensemble voting classifier (hard voting), with the above five models being individual classifiers. Provide Accuracy, Precision, Recall, F1 Score, and Classification Report.


In [43]:
from sklearn.ensemble import VotingClassifier

voting_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", VotingClassifier(
        estimators=[
            ("linear_regression", LinearRegressionClassifier()),
            ("svm", SVC(kernel="rbf", random_state=42)),
            ("decision_tree", DecisionTreeClassifier(random_state=42)),
            ("random_forest", RandomForestClassifier(n_estimators=100, random_state=42)),
            ("knn", KNeighborsClassifier(n_neighbors=5))
        ],voting="hard"
    ))
])

voting_model.fit(X_train, y_train)
y_pred_voting = voting_model.predict(X_test)

voting_results = pd.DataFrame([{
    "Model": "Hard Voting Classifier",
    "Accuracy": accuracy_score(y_test, y_pred_voting),
    "Precision": precision_score(y_test, y_pred_voting),
    "Recall": recall_score(y_test, y_pred_voting),
    "F1 Score": f1_score(y_test, y_pred_voting)
}])

print("Hard voting classifier result:")
display(voting_results.round(4).style.hide(axis="index"))

print("\nClassification Report - Hard Voting Classifier")
voting_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_voting,
        target_names=["Rejected", "Approved"],
        output_dict=True
    )
).round(3)
display(voting_report)


Hard voting classifier result:


Model,Accuracy,Precision,Recall,F1 Score
Hard Voting Classifier,0.963700,0.971700,0.969900,0.970800



Classification Report - Hard Voting Classifier


,Rejected,Approved,accuracy,macro avg,weighted avg
precision,0.951,0.972,0.964,0.961,0.964
recall,0.954,0.970,0.964,0.962,0.964
f1-score,0.952,0.971,0.964,0.961,0.964
support,323.000,531.000,0.964,854.000,854.000


### Answer 6:
1. I use pipeline to process the data(similar as Question 3)
2. I apply Hard Voting Classifier to make the result more accurate

## 7. Best Model

Among all the models tested so far, which one has the best performance? Explain your answer.


In [44]:
all_results = pd.concat([results_df, voting_results], ignore_index=True)
all_results = all_results.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

print("All model results, sorted by F1 Score:")
display(all_results.round(4))

best_model = all_results.iloc[0]
print("Best model:", best_model["Model"])
print("Best F1 Score:", round(best_model["F1 Score"], 4))
print("Best Accuracy:", round(best_model["Accuracy"], 4))

print(f"The best model is {best_model['Model']} because it has the highest F1 Score. "
    "I use F1 Score as the main measure because it balances precision and recall, "
    "which is useful for a classification problem like loan approval.")

All model results, sorted by F1 Score:


,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.9801,0.9831,0.9849,0.9840
1,Decision Tree,0.9789,0.9831,0.9831,0.9831
2,Hard Voting Classifier,0.9637,0.9717,0.9699,0.9708
3,SVM,0.9461,0.9584,0.9548,0.9566
4,Linear Regression,0.9286,0.9537,0.9303,0.9418
5,KNN,0.8958,0.9108,0.9228,0.9167


Best model: Random Forest
Best F1 Score: 0.984
Best Accuracy: 0.9801
The best model is Random Forest because it has the highest F1 Score. I use F1 Score as the main measure because it balances precision and recall, which is useful for a classification problem like loan approval.


### Answer 7: 
* I compared all model results and selected the model with the highest F1 score as the best model. I chose F1 score because it gives a balanced view of precision and recall.